# Silver models

In [ ]:
from pyspark.sql import functions as F


def select_typed_columns(df, target_schema):
    typed_columns = [
        F.col(field.name).try_cast(field.dataType).alias(field.name)
        for field in target_schema.fields
    ]

    return df.select(
        *typed_columns,
        *INGESTION_METADATA_COLUMNS,
    )


def drop_snapshot_duplicates(df, *business_keys):
    deduplication_keys = [
        *business_keys,
        "_source_snapshot_date",
    ]

    return df.dropDuplicates(deduplication_keys)

## IMDb

### Normalize null markers

In [ ]:
from pyspark.sql.types import StringType

imdb_null_normalized_dfs = {}

for table_name in IMDB_BRONZE_TABLES:
    bronze_df = spark.table(table_name).filter(
        F.col("_source_snapshot_date") == F.lit(IMDB_SNAPSHOT_DATE)
    )
    normalized_columns = [
        (
            F.when(F.col(field.name) == r"\N", F.lit(None))
            .otherwise(F.col(field.name))
            .alias(field.name)
        )
        if (
            isinstance(field.dataType, StringType)
            and field.name not in INGESTION_METADATA_COLUMNS
        )
        else F.col(field.name)
        for field in bronze_df.schema.fields
    ]

    table_name = table_name.rsplit(".", 1)[-1]
    imdb_null_normalized_dfs[table_name] = (
        bronze_df.select(*normalized_columns)
    )

### Apply explicit schemas

In [ ]:
from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
)

imdb_silver_schemas = {
    "imdb_name_basics": StructType([
        StructField("nconst", StringType(), False),
        StructField("primaryName", StringType(), True),
        StructField("birthYear", IntegerType(), True),
        StructField("deathYear", IntegerType(), True),
        StructField("primaryProfession", StringType(), True),
        StructField("knownForTitles", StringType(), True),
    ]),
    "imdb_title_basics": StructType([
        StructField("tconst", StringType(), False),
        StructField("titleType", StringType(), False),
        StructField("primaryTitle", StringType(), False),
        StructField("originalTitle", StringType(), False),
        StructField("isAdult", BooleanType(), False),
        StructField("startYear", IntegerType(), True),
        StructField("endYear", IntegerType(), True),
        StructField("runtimeMinutes", IntegerType(), True),
        StructField("genres", StringType(), True),
    ]),
    "imdb_title_crew": StructType([
        StructField("tconst", StringType(), False),
        StructField("directors", StringType(), True),
        StructField("writers", StringType(), True),
    ]),
    "imdb_title_principals": StructType([
        StructField("tconst", StringType(), False),
        StructField("ordering", IntegerType(), False),
        StructField("nconst", StringType(), False),
        StructField("category", StringType(), False),
        StructField("job", StringType(), True),
        StructField("characters", StringType(), True),
    ]),
    "imdb_title_ratings": StructType([
        StructField("tconst", StringType(), False),
        StructField("averageRating", DoubleType(), False),
        StructField("numVotes", LongType(), False),
    ]),
}

imdb_typed_dfs = {}

for table_name, normalized_df in imdb_null_normalized_dfs.items():
    imdb_typed_dfs[table_name] = select_typed_columns(
        normalized_df,
        imdb_silver_schemas[table_name],
    )

### Normalize multi-value columns into bridge tables

In [ ]:
from pyspark.sql.types import ArrayType, StringType


def explode_delimited_bridge(
    df,
    parent_key,
    source_column,
    value_column,
    delimiter=",",
):
    bridge_df = (
        df
        .where(F.col(source_column).isNotNull())
        .select(
            F.col(parent_key),
            F.explode(
                F.split(F.col(source_column), delimiter)
            ).alias(value_column),
            *INGESTION_METADATA_COLUMNS,
        )
        .withColumn(value_column, F.trim(F.col(value_column)))
        .filter(
            F.col(value_column).isNotNull()
            & (F.col(value_column) != "")
        )
    )

    return drop_snapshot_duplicates(
        bridge_df,
        parent_key,
        value_column,
    )


name_basics_df = imdb_typed_dfs["imdb_name_basics"]

imdb_name_df = name_basics_df.drop(
    "primaryProfession",
    "knownForTitles",
)

imdb_name_profession_df = explode_delimited_bridge(
    name_basics_df,
    parent_key="nconst",
    source_column="primaryProfession",
    value_column="profession",
)

imdb_name_known_for_title_df = explode_delimited_bridge(
    name_basics_df,
    parent_key="nconst",
    source_column="knownForTitles",
    value_column="tconst",
)

title_basics_df = imdb_typed_dfs["imdb_title_basics"]

imdb_title_df = title_basics_df.drop("genres")

imdb_title_genre_df = explode_delimited_bridge(
    title_basics_df,
    parent_key="tconst",
    source_column="genres",
    value_column="genre",
)

title_crew_df = imdb_typed_dfs["imdb_title_crew"]

director_df = (
    explode_delimited_bridge(
        title_crew_df,
        parent_key="tconst",
        source_column="directors",
        value_column="nconst",
    )
    .withColumn("crew_role", F.lit("director"))
)

writer_df = (
    explode_delimited_bridge(
        title_crew_df,
        parent_key="tconst",
        source_column="writers",
        value_column="nconst",
    )
    .withColumn("crew_role", F.lit("writer"))
)

imdb_title_crew_member_df = (
    director_df
    .unionByName(writer_df)
    .select(
        "tconst",
        "nconst",
        "crew_role",
        *INGESTION_METADATA_COLUMNS,
    )
)

title_principals_df = imdb_typed_dfs["imdb_title_principals"]
imdb_title_principal_df = title_principals_df.drop("characters")

imdb_title_principal_character_df = (
    title_principals_df
    .where(F.col("characters").isNotNull())
    .withColumn(
        "_parsed_characters",
        F.from_json(
            F.col("characters"),
            ArrayType(StringType()),
        ),
    )
    .select(
        "tconst",
        "ordering",
        F.explode("_parsed_characters").alias("character"),
        *INGESTION_METADATA_COLUMNS,
    )
    .withColumn("character", F.trim(F.col("character")))
    .filter(
        F.col("character").isNotNull()
        & (F.col("character") != "")
    )
)

imdb_title_principal_character_df = drop_snapshot_duplicates(
    imdb_title_principal_character_df,
    "tconst",
    "ordering",
    "character",
)

imdb_name_df = drop_snapshot_duplicates(imdb_name_df, "nconst")
imdb_title_df = drop_snapshot_duplicates(imdb_title_df, "tconst")
imdb_title_crew_member_df = drop_snapshot_duplicates(
    imdb_title_crew_member_df,
    "tconst",
    "nconst",
    "crew_role",
)
imdb_title_principal_df = drop_snapshot_duplicates(
    imdb_title_principal_df,
    "tconst",
    "ordering",
)
imdb_title_rating_df = drop_snapshot_duplicates(
    imdb_typed_dfs["imdb_title_ratings"],
    "tconst",
)

imdb_silver_dfs = {
    "imdb_name": imdb_name_df,
    "imdb_name_profession": imdb_name_profession_df,
    "imdb_name_known_for_title": imdb_name_known_for_title_df,
    "imdb_title": imdb_title_df,
    "imdb_title_genre": imdb_title_genre_df,
    "imdb_title_crew_member": imdb_title_crew_member_df,
    "imdb_title_principal": imdb_title_principal_df,
    "imdb_title_principal_character": (
        imdb_title_principal_character_df
    ),
    "imdb_title_rating": imdb_title_rating_df,
}

## MovieLens

In [ ]:
from pyspark.sql.types import (
    DoubleType,
    LongType,
    StringType,
    StructField,
    StructType,
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

movielens_silver_schemas = {
    "movielens_links": StructType([
        StructField("movieId", LongType(), False),
        StructField("imdbId", LongType(), True),
        StructField("tmdbId", LongType(), True),
    ]),
    "movielens_movies": StructType([
        StructField("movieId", LongType(), False),
        StructField("title", StringType(), False),
        StructField("genres", StringType(), True),
    ]),
    "movielens_ratings": StructType([
        StructField("userId", LongType(), False),
        StructField("movieId", LongType(), False),
        StructField("rating", DoubleType(), False),
        StructField("timestamp", LongType(), False),
    ]),
    "movielens_tags": StructType([
        StructField("userId", LongType(), False),
        StructField("movieId", LongType(), False),
        StructField("tag", StringType(), False),
        StructField("timestamp", LongType(), False),
    ]),
}

movielens_typed_dfs = {}

for table_name, target_schema in movielens_silver_schemas.items():
    bronze_df = spark.table(f"{BRONZE_NAMESPACE}.{table_name}").filter(
        F.col("_source_snapshot_date") == F.lit(MOVIELENS_SNAPSHOT_DATE)
    )

    movielens_typed_dfs[table_name] = select_typed_columns(
        bronze_df,
        target_schema,
    )

movies_df = movielens_typed_dfs["movielens_movies"]
release_year_text = F.regexp_extract(
    F.col("title"),
    r"\((\d{4})\)$",
    1,
)

movielens_movie_df = (
    movies_df
    .withColumn("title", F.trim(F.col("title")))
    .withColumn(
        "releaseYear",
        F.when(release_year_text != "", release_year_text.cast("int")),
    )
    .drop("genres")
)
movielens_movie_df = drop_snapshot_duplicates(
    movielens_movie_df,
    "movieId",
)

movielens_movie_genre_df = explode_delimited_bridge(
    movies_df.where(
        F.col("genres").isNotNull()
        & (F.col("genres") != "(no genres listed)")
    ),
    parent_key="movieId",
    source_column="genres",
    value_column="genre",
    delimiter=r"\|",
)

links_df = movielens_typed_dfs["movielens_links"]
imdb_id_text = F.col("imdbId").cast("string")
padded_imdb_id = F.when(
    F.length(imdb_id_text) < 7,
    F.lpad(imdb_id_text, 7, "0"),
).otherwise(imdb_id_text)

movielens_link_df = (
    links_df
    .withColumn(
        "tconst",
        F.when(
            F.col("imdbId") > 0,
            F.concat(F.lit("tt"), padded_imdb_id),
        ),
    )
    .select(
        "movieId",
        "tconst",
        "tmdbId",
        *INGESTION_METADATA_COLUMNS,
    )
)
movielens_link_df = drop_snapshot_duplicates(
    movielens_link_df,
    "movieId",
)

movielens_rating_df = (
    movielens_typed_dfs["movielens_ratings"]
    .withColumn(
        "ratedAt",
        F.to_timestamp(F.from_unixtime(F.col("timestamp"))),
    )
    .select(
        "userId",
        "movieId",
        "rating",
        "ratedAt",
        *INGESTION_METADATA_COLUMNS,
    )
)
movielens_rating_df = drop_snapshot_duplicates(
    movielens_rating_df,
    "userId",
    "movieId",
    "ratedAt",
)

movielens_tag_df = (
    movielens_typed_dfs["movielens_tags"]
    .withColumn("tag", F.trim(F.col("tag")))
    .withColumn(
        "taggedAt",
        F.to_timestamp(F.from_unixtime(F.col("timestamp"))),
    )
    .select(
        "userId",
        "movieId",
        "tag",
        "taggedAt",
        *INGESTION_METADATA_COLUMNS,
    )
)
movielens_tag_df = drop_snapshot_duplicates(
    movielens_tag_df,
    "userId",
    "movieId",
    "tag",
    "taggedAt",
)

movielens_silver_dfs = {
    "movielens_movie": movielens_movie_df,
    "movielens_movie_genre": movielens_movie_genre_df,
    "movielens_link": movielens_link_df,
    "movielens_rating": movielens_rating_df,
    "movielens_tag": movielens_tag_df,
}

## Persist Silver tables

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_NAMESPACE}")

silver_dfs = {
    **imdb_silver_dfs,
    **movielens_silver_dfs,
}

for table_name, silver_df in silver_dfs.items():
    target_table = f"{SILVER_NAMESPACE}.{table_name}"
    snapshot_date = SNAPSHOT_DATES[table_name.split("_", 1)[0]]
    silver_snapshot_df = silver_df.filter(
        F.col("_source_snapshot_date") == F.lit(snapshot_date)
    )

    (
        silver_snapshot_df.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", snapshot_predicate(snapshot_date))
        .saveAsTable(target_table)
    )

## Silver quality checks

In [ ]:
movie_keys_df = movielens_movie_df.select(
    "movieId",
    "_source_snapshot_date",
).distinct()

quality_checks = [
    (
        "movielens_movie_null_id",
        movielens_movie_df.filter(F.col("movieId").isNull()).count(),
    ),
    (
        "movielens_link_invalid_imdb_id",
        links_df.filter(
            F.col("imdbId").isNotNull()
            & (F.col("imdbId") <= 0)
        ).count(),
    ),
    (
        "movielens_rating_invalid_value",
        movielens_rating_df.filter(
            F.col("rating").isNull()
            | (F.col("rating") < 0.5)
            | (F.col("rating") > 5.0)
        ).count(),
    ),
    (
        "movielens_rating_unparsed_timestamp",
        movielens_rating_df.filter(
            F.col("ratedAt").isNull()
        ).count(),
    ),
    (
        "movielens_tag_unparsed_timestamp",
        movielens_tag_df.filter(
            F.col("taggedAt").isNull()
        ).count(),
    ),
    (
        "movielens_rating_missing_movie",
        movielens_rating_df.join(
            movie_keys_df,
            ["movieId", "_source_snapshot_date"],
            "left_anti",
        ).count(),
    ),
    (
        "movielens_tag_missing_movie",
        movielens_tag_df.join(
            movie_keys_df,
            ["movieId", "_source_snapshot_date"],
            "left_anti",
        ).count(),
    ),
    (
        "movielens_link_missing_movie",
        movielens_link_df.join(
            movie_keys_df,
            ["movieId", "_source_snapshot_date"],
            "left_anti",
        ).count(),
    ),
    (
        "imdb_title_null_id",
        imdb_title_df.filter(F.col("tconst").isNull()).count(),
    ),
    (
        "imdb_name_null_id",
        imdb_name_df.filter(F.col("nconst").isNull()).count(),
    ),
]

silver_quality_summary_df = spark.createDataFrame(
    quality_checks,
    ["check_name", "failed_row_count"],
)

display(silver_quality_summary_df.orderBy("check_name"))